In [ ]:
import pandas as pd

npm_mttu_mttr = pd.read_csv("npm_mttu_mttr.csv")
pypi_mttu_mttr = pd.read_csv("pypi_mttu_mttr.csv")

all_mttu_mttr = pd.concat([npm_mttu_mttr, pypi_mttu_mttr], ignore_index=True)
sc_data = pd.read_csv("../../data/local_nonlocal_checks_with_missing_values.csv")

## Mismatch version numbers 
MTTU / MTTR is calculated using deps.dev, and the version numbers come from deps.dev \
We collected Scorecard data using version numbers (release tags) from GitHub Rest API 

The deps.dev versions do not always the same as the version release tags in GitHub. 

**For example:**

**package name: 0g** \
Version number in GitHub (as listed in the SC data) - 0g@0.2.2 

Version number in deps.devs (as listed in the MTTU / MTTR data) - 0.2.2 


**Solution: remove any characters *preceding* the semantic version in the Scorecard data, and merge on that**

In the following code chunk, remove any prefixes before version number. 

In [ ]:
import re

def clean_version(tag_name):
    """Remove package name prefixes before version numbers."""
    tag_name = str(tag_name)
    
    # Strategy 1: Find @ followed by a proper version pattern (digit.digit...)
    # Handles: @org/package@1.0.0 -> 1.0.0
    matches = list(re.finditer(r'@(\d+\.\d+[^\s]*)$', tag_name))
    if matches:
        return matches[-1].group(1)
    
    # Strategy 2: Find / followed by version pattern (for package/version style)
    # Handles: package-name/v1.0.0 -> 1.0.0
    match = re.search(r'/[v]?(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 3: Find _ followed by v and version pattern
    # Handles: gmv3_v2.0.0 -> 2.0.0
    match = re.search(r'_v(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 4: Handle package-version style with hyphen
    # Handles: package-1.0.0 -> 1.0.0
    match = re.search(r'-(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 5: Fallback - remove any non-digit prefix (for v1.0.0 style)
    # Handles: v1.0.0 -> 1.0.0
    match = re.search(r'^[^\d]*(\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    return tag_name

In the following code, test for semantic versioning.

In [ ]:
def is_semantic_version(tag_name):
    """
    Check if tag follows semantic versioning: MAJOR.MINOR.PATCH
    Also accepts minor versions (MAJOR.MINOR) and major versions (MAJOR)
    Allows suffixes like alpha, beta, rc, post, dev, etc.
    """
    tag_name = str(tag_name)
    
    # Pattern for semantic versioning with optional suffixes
    # Matches: X.Y.Z, X.Y, or X (where X, Y, Z are numbers)
    # Allows suffixes: -alpha, .post1, a0, b1, rc2, dev3, etc.
    pattern = r'^\d+(\.\d+)?(\.\d+)?([a-zA-Z0-9\.\-]+)?$'
    
    return bool(re.match(pattern, tag_name))

Apply the data cleaning (clean_version) to the Scorecard version numbers and investigate semantic versioning after applying the function.

In [ ]:
# Apply the updated clean_version function
sc_data['tag_name_clean'] = sc_data['tag_name'].apply(clean_version)

# Check semantic versioning
non_semver = sc_data[~sc_data['tag_name_clean'].apply(is_semantic_version)]

print(f"Total tags: {len(sc_data)}")
print(f"Number of non-semantic version tags: {len(non_semver)}")
print(f"Percentage: {len(non_semver)/len(sc_data)*100:.2f}%")

# Show examples
print(f"\nExamples of remaining non-semantic version tags:")
print(non_semver['tag_name_clean'])

In [ ]:
## Now merge the cleaned version numbers with the version numbers in mttu_mttr data and check how many records we have after the merge
merged_data = pd.merge(all_mttu_mttr, 
                       sc_data,
                       left_on=['package_name', 'package_version'], 
                       right_on=['package_name', 'tag_name_clean'], 
                       how='inner')
print(f"\nTotal records after merging with cleaned version numbers: {len(merged_data)}")


In [ ]:
final_merged_data = merged_data.drop(columns=['error', 'Maintained_details'])

In [ ]:
final_merged_data.to_csv("sc_mttu_mttr_final.csv", index=False)